## 可视化流程图

In [ ]:
初始状态（4个节点，每个节点有完整数据）：
Node 0: [a0, a1, a2, a3]
Node 1: [b0, b1, b2, b3]
Node 2: [c0, c1, c2, c3]
Node 3: [d0, d1, d2, d3]

目标：每个节点都得到 [a0+b0+c0+d0, a1+b1+c1+d1, a2+b2+c2+d2, a3+b3+c3+d3]

## 阶段1：Scatter-Reduce（N-1步）

In [ ]:
def scatter_reduce_phase(data_blocks,rank,world_size):
    '''
    scatter-reduce阶段：每个节点累加一部分数据
    
    假设world_size = 4 ,数据被分成4块
    
    '''
    # 初始化状态
    # step 0 ：每个节点发送一块给下一个节点，接收来自上一个节点的块，并累加
    
    for step in range(world_size - 1):
        # 计算发送 和 接收 的块索引
        send_block_idx = (rank - step) % world_size # 发送给下一个节点
        recv_block_idx = (rank - step - 1) % world_size # 接收上个节点的块并累加
        
    # 最终结果：
    # Node 0: [a0+b0+c0+d0, a1, a2, a3]  # 块0完全聚合
    # Node 1: [a0, a1+b1+c1+d1, a2, a3]  # 块1完全聚合
    # Node 2: [a0, a1, a2+b2+c2+d2, a3]  # 块2完全聚合
    # Node 3: [a0, a1, a2, a3+b3+c3+d3]  # 块3完全聚合

## 阶段2：AllGather（N-1步）

In [ ]:
def  allgather_phase(data_blocks,rank,world_size):
    '''
    AllGather阶段：将聚合后的块广播给所有节点
    '''
    
    # 从scatter-reduce阶段结束开始，每个节点已经拥有一个完整的聚合块
    
    for step in range(world_size - 1):
        # 发送自己拥有的完全聚合块，给下一个节点
        # 接收上一个节点发送的完全聚合块
        
    # 最终结果：所有节点都拥有所有完全聚合的块
    
    
    

## 完整实例

In [ ]:
Step 0:
  Node 0 -> Node 1: send block 0, receive block 3
  Node 1 -> Node 2: send block 1, receive block 0
  Node 2 -> Node 3: send block 2, receive block 1
  Node 3 -> Node 0: send block 3, receive block 2

Step 1:
  Node 0 -> Node 1: send block 3, receive block 2
  Node 1 -> Node 2: send block 0, receive block 3
  Node 2 -> Node 3: send block 1, receive block 0
  Node 3 -> Node 0: send block 2, receive block 1

Step 2:
  Node 0 -> Node 1: send block 2, receive block 1
  Node 1 -> Node 2: send block 3, receive block 2
  Node 2 -> Node 3: send block 0, receive block 3
  Node 3 -> Node 0: send block 1, receive block 0

In [ ]:
import torch
import torch.distributed as dist
import numpy as np

def ring_allreduce_example():
    '''
    Ring AllReduce 的详细实现
    '''
    
    # 模拟4个节点，每个节点8个梯度值
    world_size = 4
    data_size = 8
    
    # 为每个节点生成随机数据（模拟梯度）
    gradients = []
    for i in range(world_size):
        grad = torch.tensor((i+1)*10 +j for j in range(data_size),dtype = torch.float32)
        gradients.append(grad)
        print(f"Node {i} initial:{grad}")
    
    print("\n"+"="*60)
    print('Starting RingReduce')
    print('='*60)
    
    # 将数据分成world_size块
    chunk_size = data_size // world_size # 8 // 4 = 2
    chunks = []
    for i in range(world_size):
        node_chunks = []
        for j in range(world_size):
            start = j * chunk_size
            end = (j+1) * chunk_size
            node_chunks.append(gradients[i][start:end].clone()) # 切分数据
        chunks.append(node_chunks) 
    
    # 展示初始分块
    print("\n=== phase 0:Initial Chunking ===")
    for i in range(world_size):
        print(f"Node {i}:{chunks[i][j].tolist()} for j in range(world_size)")
        
        
    # === 阶段1 ：scatter-reduce ===
    print('\n === Phase 1: Scatter-Reduce ===')
    
    # 创建本地数据副本
    local_chunks = [ chunks[i].copy() for i in range(world_size)]
    
    for step in range(world_size - 1):
        print(f"\n --- Step {step+1} of scatter-Reduce ---")
        
        '''
        每个step将所有的node执行一次，发送和接收
        '''
        for rank in range(world_size):
            # 计算发送和接收的块索引
            send_idx = (rank - step) % world_size
            recv_idx = (rank - step - 1) %world_size
            
            # 模拟发送和接收
            sender = rank                       # 发送数据的node编号
            receiver = (rank + 1) % worlds_size # 接收数据的node编号
            
            # 发送数据
            send_data = local_chunks[sender][send_idx].clone()
            
            # 接收数据并累加
            recv_data = local_chunks[receiver][recv_idx].clone()
            
            # 将接收到的数据累加
            local_chunks[receiver][recv_idx] = send_data + recv_data
            
            print(f"  Node {sender} -> Node {receiver}: "
                  f"sending block {send_idx} ({send_data.tolist()}), "
                  f"node {receiver} updates block {recv_idx} to {local_chunks[receiver][recv_idx].tolist()}")
     # Scatter-Reduce 完成后的状态
    print("\n === After Scatter-Redece ===")
    for i in range(world_size):
        print(f"Node {i} : {[ local_chunks[i][j].tolist() for j in range(world_size)]}")
        
    # === 阶段2：AllGather ===
    print('\n ===Phase 2: AllGather===')
    
    for step in range(world_size -1):
        print(f"\n--- Step {step+1} of AllGather ---")
        
        for rank in range(world_size):
            # 计算要发送的块索引（已经聚合好的块）
            send_idx = (rank - step - 1) % world_size
            
            sender = rank
            receiver = (rank + 1) % world_size
            
            # 发送聚合好的块
            send_data = local_chunks[sender][send_idx].clone()
            
            # 接收并覆盖（不需要累加，因为已经是完整结果）
            local_chunks[receiver][send_idx] = send_data
            
            print(f"  Node {sender} -> Node {receiver}: "
                  f"broadcasting block {send_idx} ({send_data.tolist()}), "
                  f"node {receiver} updates block {send_idx} to {send_data.tolist()}")
    # 最终结果
    print("\n=== Final Result After Ring AllReduce ===")
    for i in range(world_size):
        final_result = torch.cat(local_chunks[i])
        print(f"Node {i}: {final_result.tolist()}")
    
    # 验证结果
    expected = sum(gradients)
    print(f"\nExpected sum: {expected.tolist()}")
    print(f"All nodes match expected: {all(torch.allclose(torch.cat(local_chunks[i]), expected) for i in range(world_size))}")

# 运行示例
ring_allreduce_example()   
    

## 在pytorch中的应用

In [ ]:
import torch
import torch.distributed as dist
import torch.multiprocessing as mp

def ring_allreduce_demo(rank,world_size):
    '''
    使用pytorch的dist.all_reduce演示 Ring AllReduce
    '''
    
    # 初始化进程
    dist.init_process_group(
        backend = 'gloo' , # 使用gloo后端（CPU）
        init_method='tcp://127.0.0.1:29500',
        rank=rank,
        world_size=world_size
    )
    
    # 创建梯度张量
    grad = torch.tensor([rank+1.0,(rank+1)*2],dtype=torch.float32)
    print(f"Rank {rank} initial grad:{grad}")
    
    # 执行allreduce（默认求和）
    dist.all_reduce(grad,op=dist.ReduceOp.SUM)
    
    # 除以world_size 得到平均值
    grad /=wold_size
    
    print(f"Rank {rank} after all_reduce :{grad}")
    
    # 清理
    dist.destroy_process_group()
    
def run_ring_allreduce_demo():
    world_size = 4
    mp.spawn(ring_allreduce_demo,args=(world_size,),nprocs = world_size)
    
# run_ring_allreduce_demo() # 需要多进程环境

## 分层 Ring AllReduce（Hierarchical Ring）

In [ ]:
def hierarchical_ring_allreduce():
    """
    分层 Ring AllReduce：先在一个机箱内聚合，再跨机箱聚合
    """
    # 假设有 2 个节点，每个节点 4 个 GPU
    num_nodes = 2
    gpus_per_node = 4
    world_size = num_nodes * gpus_per_node
    
    # 第一层：（节点内）聚合（使用 NVLink，高速）
    # 每个节点内使用 Ring AllReduce
    
    # 第二层：（节点间）聚合（使用 InfiniBand/以太网）
    # 只发送聚合后的数据，减少跨节点通信量
    
    print(f"Hierarchical Ring AllReduce:")
    print(f"  Intra-node: {gpus_per_node} GPUs, using NVLink")
    print(f"  Inter-node: {num_nodes} nodes, using network")
    print(f"  Total communication reduced by factor of {gpus_per_node}")

#### 实际应用：NCCL、Horovod、PyTorch DDP 默认使用 Ring AllReduce

#### 现代优化：结合 NVLink、InfiniBand 等高速互联技术